In [24]:
import torch, tqdm, time, os, json
import numpy as np
import pandas as pd
import pytorch_lightning as pl
from models.est_model import GroundPressureModel
from torchmetrics import Accuracy
from torch.utils.data import DataLoader, Dataset, random_split
from pytorch_lightning.callbacks import DeviceStatsMonitor, EarlyStopping
from sklearn.metrics import r2_score

In [3]:
monitor = DeviceStatsMonitor()

In [4]:
# set train setting
save_model = True
train_num_epoch = 50000
min_loss = 100
dl_workers = 0

In [5]:
gpu = torch.device('cuda')
torch.set_float32_matmul_precision('high')

In [6]:
# load hyperparameter of json file
with open('.' + os.sep + os.path.join('models', 'params_dnn_20220207-012403.json'), 'r') as file:
    hyper_params = json.load(file)

n_input = hyper_params['n_of_inputs']
h1_neuron = hyper_params['h1_neuron']
h2_neuron = hyper_params['h2_neuron']
n_output = hyper_params['n_of_outputs']

In [7]:
model = GroundPressureModel(n_input, h1_neuron, h2_neuron, n_output)
print(model)

GroundPressureModel(
  (model): Sequential(
    (0): Linear(in_features=4, out_features=1239, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1239, out_features=1128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=1128, out_features=6, bias=True)
  )
  (loss_func): MSELoss()
)


In [31]:
data = pd.read_csv('./resources/sim_data.csv')
feature_names = ['lift_weight(ton)', 'lift_height(m)', 'rising_angle(deg)', 'swing_angle(deg)']
target_names = ['left_ground_pressure_min(kg/cm2)', 'left_ground_pressure_max(kg/cm2)', 'left_pressure_length(m)', 'right_ground_pressure_min(kg/cm2)', 'right_ground_pressure_max(kg/cm2)', 'right_pressure_length(m)']

feature_data = []
target_data = []

for feature_name in feature_names:
    feature_data.append(data[feature_name])

for target_name in target_names:
    target_data.append(data[target_name])

feature_data = np.array(feature_data, dtype=np.float32).T
target_data = np.array(target_data, dtype=np.float32).T

class TensorDataset(Dataset):
    def __init__(self, feature, target):
        self.x_data = torch.tensor(feature)
        self.y_data = torch.tensor(target)
        self.len = self.x_data.shape[0]

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.len

train_dataset = TensorDataset(feature_data, target_data)
train_data_loader = DataLoader(train_dataset, batch_size=len(train_dataset), num_workers=20)

In [32]:
# train model
early_stop_callback = EarlyStopping(monitor='train_loss', mode='min', verbose=True, min_delta=0.01, patience=20)
trainer = pl.Trainer(callbacks=[early_stop_callback], accelerator='gpu', enable_progress_bar=True, max_epochs=200)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [33]:
trainer.fit(model=model, train_dataloaders=train_data_loader)

/mnt/workspace/estimate_crawler_crane_ground_pressure/venv/lib/python3.8/site-packages/pytorch_lightning/trainer/configuration_validator.py:72: PossibleUserWarning: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
  rank_zero_warn(
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
/mnt/workspace/estimate_crawler_crane_ground_pressure/venv/lib/python3.8/site-packages/pytorch_lightning/trainer/configuration_validator.py:72: PossibleUserWarning: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
  rank_zero_warn(
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 2 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 1 - CUDA_VISIBLE

Epoch 0: 100%|██████████| 1/1 [00:01<00:00,  1.63s/it, v_num=51]

[rank: 0] Metric train_loss improved. New best score: 0.048
[rank: 1] Metric train_loss improved. New best score: 0.039


Epoch 20: 100%|██████████| 1/1 [00:00<00:00,  1.03it/s, v_num=51]

[rank: 0] Monitored metric train_loss did not improve in the last 20 records. Best score: 0.048. Signaling Trainer to stop.
[rank: 1] Monitored metric train_loss did not improve in the last 20 records. Best score: 0.039. Signaling Trainer to stop.


Epoch 20: 100%|██████████| 1/1 [00:01<00:00,  1.04s/it, v_num=51]


In [17]:
torch.save(model.state_dict(), './models/est_ground_pressure.pt')

In [34]:
model.eval()
for data in train_dataset:
    print(r2_score(data[1].detach().numpy() ,model(data[0]).detach().numpy()))
    print(model(data[0]).detach().numpy())
    print(data[1].detach().numpy())

0.6225184331012806
[ 1.9065717  -0.06943983  3.7490997   2.2069085   0.21829888  3.5079384 ]
[3.97 0.   3.72 3.97 0.   3.72]
0.6255156729153468
[ 1.3904766  -0.3538997   3.793344    2.0756118   0.30254337  3.7210429 ]
[2.61 0.   4.1  4.58 0.   4.1 ]
0.9738267892034408
[ 0.9095661  -0.02818424  5.0503197   3.5360126   0.09472443  5.1174917 ]
[1.15 0.   5.59 4.13 0.   5.59]
0.9832480579751413
[ 2.4539833  -0.09166932  5.115115    2.881925    0.36110055  4.803487  ]
[2.93 0.   5.04 2.93 0.   5.04]
0.9687016280486825
[ 2.1130867  -0.5873586   5.1812444   2.743075    0.31658426  5.0907044 ]
[2.16 0.   5.34 3.36 0.   5.34]
0.9846663392379582
[ 1.2428461  -0.23777133  6.0515704   3.5174217   0.3252746   6.0267386 ]
[1.25 0.   6.53 3.27 0.   6.53]
0.9337509676191713
[ 2.9940333  -0.11439705  6.483156    3.5542185   0.50735086  6.0985265 ]
[2.21 0.   6.68 2.21 0.   6.68]
0.9441709527300511
[ 2.7286901  -0.76033115  6.553483    3.4507222   0.3778583   6.5010986 ]
[1.83 0.04 6.75 2.46 0.05 6.75]
